In [11]:
# imports and settings

import os
import gpxpy
import time
import datetime
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

import utils as ut

# do not show warnings
warnings.filterwarnings("ignore")

# autoreload for development
%load_ext autoreload
%autoreload 2

print("Imports complete.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Imports complete.


In [12]:
# folder_path = '../data/Itamar_AUV/Part1_StraightLine/IcListen6692'
# file_list = os.listdir(folder_path)
# file_list = [f for f in file_list if f.endswith('.wav')]
# file_list.sort()

# data = []
# for file in file_list:
#     # print(f"Loading file: {file}")
#     _data, fs = librosa.load(os.path.join(folder_path, file))
#     data.append(_data)
# data = np.concatenate(data)
# # data = data[80*fs:]

data, fs = librosa.load('../data/Itamar_AUV/Part1_StraightLine/IcListen6692/leg1_straight_line_2_8m_s_6922_827-837.wav')

print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")

data final shape: (14553000,) of duration: 660.0 seconds


In [13]:
# folder_path = '../data/Itamar_AUV/Part1_StraightLine/Straight_line_gpx'
# file_list = os.listdir(folder_path)
# file_list = [f for f in file_list if f.endswith('.gpx')]
# file_list.sort()

# rx_location = gpxpy.gpx.GPXWaypoint(32.855333333, 34.93238333)
# cut_time = datetime.datetime(2026, 1, 7, 8, 37, 30, tzinfo=datetime.timezone.utc)

# gps_data = []
# distances = []
# first_time = None
# last_time = None
# for file in file_list:
#     with open(os.path.join(folder_path, file), 'r') as f:
#         _gpx_data = gpxpy.parse(f)
#         _track = _gpx_data.tracks[0]
#         _segment = _track.segments[0]
#         _points = _segment.points
#         for point in _points:
#             if point.time <= cut_time:
#                 gps_data.append(point)
#                 distances.append(rx_location.distance_2d(point))
#                 if first_time is None:
#                     first_time = point.time
#                 last_time = point.time

# distances = np.array(distances)
# t_distances = np.linspace(0, len(data)/fs, len(distances))
# 
# print(f"gps data from {first_time} to {last_time}")

distance_file = '../data/Itamar_AUV/Part1_StraightLine/IcListen6692/leg1_straight_line_2_8m_s_6922_827-837_distance_log.txt'
df = pd.read_csv(distance_file, sep=' ')
df.head()
distances = df['distance_m'].values
t_distances = df['second'].values

In [14]:
nperseg = 1000
overlap = 0.5
window = 'hanning'
dc = 100
crop_freq = 16000
norm_size = 5
s2g_quantization_levels = 3
s2g_mode = 'wasserstein'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

In [15]:
# fig = make_subplots(rows=2, cols=1, shared_xaxes=True)
# F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)
# fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
# fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
# fig.update_xaxes(title_text="Time (s)", row=2, col=1)
# fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
# fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
# fig.update_layout(height=800, width=1200, title_text=f"AUV experiment")
# fig.show(renderer='browser')

In [16]:
welch_detector = ut.WelchDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
talmon_detector = ut.TalmonDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=s2g_quantization_levels)

In [17]:
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

# f0_1_welch_scores = []
# f0_2_welch_scores = []

f0_1_s2g_scores = []
f0_2_s2g_scores = []
f0_3_s2g_scores = []

f0_1_td_scores = []
f0_2_td_scores = []
f0_3_td_scores = []

f0_1 = 441  # Hz
f0_2 = 992  # Hz
f0_3 = 5556  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window
    
    # P = welch_detector.get_score_1fb(_data, f0_1)
    # f0_1_welch_scores.append(P)
    # P = welch_detector.get_score_1fb(_data, f0_2)
    # f0_2_welch_scores.append(P)

    TD = talmon_detector.get_score_1fb(_data, f0_1)
    f0_1_td_scores.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f0_2)
    f0_2_td_scores.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f0_3)
    f0_3_td_scores.append(TD)

    K = s2g_detector.get_score_1fb(_data, f0_1)
    f0_1_s2g_scores.append(K)
    K = s2g_detector.get_score_1fb(_data, f0_2)
    f0_2_s2g_scores.append(K)
    K = s2g_detector.get_score_1fb(_data, f0_3)
    f0_3_s2g_scores.append(K)


detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")


completed all detectors


In [18]:
# fig = make_subplots(rows=4, 
#                     cols=1, 
#                     row_heights=[0.7, 0.1, 0.1, 0.1], 
#                     vertical_spacing=0.01, 
#                     shared_xaxes=True,
#                     row_titles=["Spectrogram", "distance", "Talmon", "S2G"])

# fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
# fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
# # fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_welch_scores, name=f'Welch f={f0_1} Hz', marker_color='blue'), row=3, col=1)
# # fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_welch_scores, name=f'Welch f={f0_2} Hz', marker_color='cyan'), row=3, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores, name=f'Talmon f={f0_1} Hz', marker_color='blue', showlegend=False), row=3, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores, name=f'Talmon f={f0_2} Hz', marker_color='green', showlegend=False), row=3, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores, name=f'Score f={f0_1} Hz', marker_color='blue'), row=4, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores, name=f'Score f={f0_2} Hz', marker_color='green'), row=4, col=1)


# fig.update_xaxes(title_text="Time (s)", row=3, col=1)
# fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
# fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
# fig.update_yaxes(title_text="Score", row=3, col=1)
# fig.update_yaxes(title_text="Score", row=4, col=1)
# # fig.update_yaxes(title_text="Score", row=5, col=1)

# fig.update_layout(height=800, width=1200, title_text=f"Scooter experiment 600m, scores for f0 = {f0_1}, {f0_2} Hz")
# fig.show(renderer="browser")

---

## Add Noise

In [19]:
# BG Noise WAVs
folder_path = '../data/croatia/2507_1_1000m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

bg_data = []
for file in file_list[:5]:
    _data, fs = librosa.load(os.path.join(folder_path, file))
    bg_data.append(_data)
bg_data = np.concatenate(bg_data)
print(f"bg_data final shape: {bg_data.shape} of duration: {len(bg_data)/fs} seconds")

# Add BG Noise to data
noise_factor = 10
bg_in_shape = np.concatenate([bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data])
data = data + noise_factor * bg_in_shape[:len(data)]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")

bg_data final shape: (6615000,) of duration: 300.0 seconds
data final shape: (14553000,) of duration: 660.0 seconds


In [20]:
nF, nT, nSxx, nphasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

# f0_1_welch_scores_wn = []
# f0_2_welch_scores_wn = []

f0_1_td_scores_wn = []
f0_2_td_scores_wn = []
f0_3_td_scores_wn = []

f0_1_s2g_scores_wn = []
f0_2_s2g_scores_wn = []
f0_3_s2g_scores_wn = []

f0_1 = 441  # Hz
f0_2 = 992  # Hz
f0_3 = 5556  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    # P = welch_detector.get_score_1fb(_data, f0_1)
    # f0_1_welch_scores_wn.append(P)
    # P = welch_detector.get_score_1fb(_data, f0_2)
    # f0_2_welch_scores_wn.append(P)

    TD = talmon_detector.get_score_1fb(_data, f0_1)
    f0_1_td_scores_wn.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f0_2)
    f0_2_td_scores_wn.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f0_3)
    f0_3_td_scores_wn.append(TD)

    K = s2g_detector.get_score_1fb(_data, f0_1)
    f0_1_s2g_scores_wn.append(K)
    K = s2g_detector.get_score_1fb(_data, f0_2)
    f0_2_s2g_scores_wn.append(K)
    K = s2g_detector.get_score_1fb(_data, f0_3)
    f0_3_s2g_scores_wn.append(K)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")

completed all detectors


In [ ]:
fig = make_subplots(rows=4, 
                    cols=2, 
                    row_heights=[0.7, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    column_titles=["No Noise", "SNR 1:10"],
                    row_titles=["Spectrogram", "distance", "Talmon", "S2G"])

_ds = max(1, Sxx.shape[1] // 1000)

fig.add_trace(go.Heatmap(z=Sxx[:, ::_ds], x=T[::_ds], y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores, name=f'Score f={f0_1} Hz', marker_color='blue'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores, name=f'Score f={f0_2} Hz', marker_color='green'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_td_scores, name=f'Score f={f0_3} Hz', marker_color='purple'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_s2g_scores, name=f'Score f={f0_3} Hz', marker_color='purple', showlegend=False), row=4, col=1)
dix_nn_1 = np.where(detectors_taxis >= 167)[0][0]
dix_nn_2 = np.where(detectors_taxis >= 156)[0][0]
fig.add_vline(x=detectors_taxis[dix_nn_1] , line_dash="dash", line_color="red", line_width=2, name="first detection", showlegend=True, row=3, col=1)
fig.add_vline(x=detectors_taxis[dix_nn_2], line_dash="dash", line_color="red", line_width=2, row=4, col=1)

fig.add_trace(go.Heatmap(z=nSxx[:, ::_ds], x=nT[::_ds], y=nF, colorscale='viridis', showlegend=False, showscale=False), row=1, col=2)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores_wn, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=3, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores_wn, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=3, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_td_scores_wn, name=f'Score f={f0_3} Hz', marker_color='purple', showlegend=False), row=3, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores_wn, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=4, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores_wn, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=4, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_s2g_scores_wn, name=f'Score f={f0_3} Hz', marker_color='purple', showlegend=False), row=4, col=2)
dix_nn_1 = np.where(detectors_taxis >= 167)[0][0]
dix_nn_2 = np.where(detectors_taxis >= 145)[0][0]
fig.add_vline(x=detectors_taxis[dix_nn_1] , line_dash="dash", line_color="red", line_width=2, row=3, col=2)
fig.add_vline(x=detectors_taxis[dix_nn_2], line_dash="dash", line_color="red", line_width=2, row=4, col=2)


fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_xaxes(title_text="Time (s)", row=4, col=1)
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=2)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=2)
fig.update_yaxes(title_text="Score", row=3, col=2)
fig.update_yaxes(title_text="Score", row=4, col=2)
fig.update_xaxes(title_text="Time (s)", row=4, col=2)

fig.update_layout(height=800, width=1600, title_text=f"AUV leg 1 straight line, scores for f0 = {f0_1}, {f0_2}, {f0_3} (Hz)")
# fig.show(renderer="browser")
fig.write_image("../results/plots/nb85_auv_w_noise.pdf")

---
## Test Q levels

In [22]:

data, fs = librosa.load('../data/Itamar_AUV/Part1_StraightLine/IcListen6692/leg1_straight_line_2_8m_s_6922_827-837.wav')

distance_file = '../data/Itamar_AUV/Part1_StraightLine/IcListen6692/leg1_straight_line_2_8m_s_6922_827-837_distance_log.txt'
df = pd.read_csv(distance_file, sep=' ')
df.head()
distances = df['distance_m'].values
t_distances = df['second'].values

# BG Noise WAVs
folder_path = '../data/croatia/2507_1_1000m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

bg_data = []
for file in file_list[:5]:
    _data, fs = librosa.load(os.path.join(folder_path, file))
    bg_data.append(_data)
bg_data = np.concatenate(bg_data)
print(f"bg_data final shape: {bg_data.shape} of duration: {len(bg_data)/fs} seconds")

# Add BG Noise to data
noise_factor = 10
bg_in_shape = np.concatenate([bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data])
data = data + noise_factor * bg_in_shape[:len(data)]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")

bg_data final shape: (6615000,) of duration: 300.0 seconds
data final shape: (14553000,) of duration: 660.0 seconds


In [23]:
nperseg = 1000
overlap = 0.5
window = 'hanning'
dc = 100
crop_freq = 8000
norm_size = 5
s2g_mode = 'wasserstein'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

s2g2 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=2)
s2g3 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=3)
s2g5 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=5)
s2g10 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=10)
s2g20 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=20)

In [24]:
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f0_1_s2g_2 = []
f0_2_s2g_2 = []
f0_3_s2g_2 = []

f0_1_s2g_3 = []
f0_2_s2g_3 = []
f0_3_s2g_3 = []

f0_1_s2g_5 = []
f0_2_s2g_5 = []
f0_3_s2g_5 = []

f0_1_s2g_10 = []
f0_2_s2g_10 = []
f0_3_s2g_10 = []

f0_1_s2g_20 = []
f0_2_s2g_20 = []
f0_3_s2g_20 = []

f0_1 = 441  # Hz
f0_2 = 992  # Hz
f0_3 = 5556  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    K = s2g2.get_score_1fb(_data, f0_1)
    f0_1_s2g_2.append(K)
    K = s2g2.get_score_1fb(_data, f0_2)
    f0_2_s2g_2.append(K)
    K = s2g2.get_score_1fb(_data, f0_3)
    f0_3_s2g_2.append(K)

    K = s2g3.get_score_1fb(_data, f0_1)
    f0_1_s2g_3.append(K)
    K = s2g3.get_score_1fb(_data, f0_2)
    f0_2_s2g_3.append(K)
    K = s2g3.get_score_1fb(_data, f0_3)
    f0_3_s2g_3.append(K)

    K = s2g5.get_score_1fb(_data, f0_1)
    f0_1_s2g_5.append(K)
    K = s2g5.get_score_1fb(_data, f0_2)
    f0_2_s2g_5.append(K)
    K = s2g5.get_score_1fb(_data, f0_3)
    f0_3_s2g_5.append(K)

    K = s2g10.get_score_1fb(_data, f0_1)
    f0_1_s2g_10.append(K)
    K = s2g10.get_score_1fb(_data, f0_2)
    f0_2_s2g_10.append(K)
    K = s2g10.get_score_1fb(_data, f0_3)
    f0_3_s2g_10.append(K)

    K = s2g20.get_score_1fb(_data, f0_1)
    f0_1_s2g_20.append(K)
    K = s2g20.get_score_1fb(_data, f0_2)
    f0_2_s2g_20.append(K)
    K = s2g20.get_score_1fb(_data, f0_3)
    f0_3_s2g_20.append(K)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")

completed all detectors


In [ ]:
fig = make_subplots(rows=7, 
                    cols=1, 
                    row_heights=[0.6, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "Distance", "Q=2","Q=3","Q=5","Q=10","Q=20"])

_ds = max(1, Sxx.shape[1] // 1000)
fig.add_trace(go.Heatmap(z=Sxx[:, ::_ds], x=T[::_ds], y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_2, name=f'Score f={f0_1} Hz', marker_color='blue'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_2, name=f'Score f={f0_2} Hz', marker_color='green'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_s2g_2, name=f'Score f={f0_3} Hz', marker_color='purple'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_3, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_3, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_s2g_3, name=f'Score f={f0_3} Hz', marker_color='purple', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_5, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=5, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_5, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=5, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_s2g_5, name=f'Score f={f0_3} Hz', marker_color='purple', showlegend=False), row=5, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_10, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=6, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_10, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=6, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_s2g_10, name=f'Score f={f0_3} Hz', marker_color='purple', showlegend=False), row=6, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_20, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=7, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_20, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=7, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_3_s2g_20, name=f'Score f={f0_3} Hz', marker_color='purple', showlegend=False), row=7, col=1)



fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_yaxes(title_text="Score", row=5, col=1)
fig.update_yaxes(title_text="Score", row=6, col=1)
fig.update_yaxes(title_text="Score", row=7, col=1)
fig.update_xaxes(title_text="Time (s)", row=7, col=1)

fig.update_layout(height=800, width=1200, title_text=f"Scooter experiment 600m, scores for f0 = {f0_1}, {f0_2} Hz")
# fig.show(renderer="browser")
fig.write_image("../results/plots/nb85_auv_Qanalysis.pdf")